# Zip and ZCTA Crosswalk 

ZIP codes and ZIP Code Tabulation Areas (ZCTAs) are both used for spatial aggregation and are associated with postal address systems in the United States. However, there are important differences between the two.

#### Purpose:
ZIP codes (Zone Improvement Plan codes) were initially introduced by the United States Postal Service (USPS) in 1963 to improve mail sorting and delivery efficiency. They are primarily designed for postal purposes, focusing on the efficient routing of mail.

ZCTAs, on the other hand, were created by the U.S. Census Bureau to facilitate census data tabulation and analysis. ZCTAs were introduced in the late 1990s to overcome the limitations of ZIP codes for statistical purposes, such as population and demographic analysis.

#### Geographic Coverage:
ZIP codes are based on the postal delivery routes established by the USPS. They typically represent smaller geographic areas, such as individual buildings, streets, or parts of a city. ZIP codes can be continuously changing, with new codes added and existing ones modified to accommodate changes in postal operations.

ZCTAs, on the other hand, are created by aggregating census blocks based on the most frequently occurring ZIP codes within each block. ZCTAs often represent larger areas, such as cities, towns, or even portions of counties. They remain relatively stable over time and are not affected by changes in postal operations.

#### Boundaries:
ZIP codes do not have official boundaries. Instead, they are based on the USPS delivery routes and can sometimes overlap or have irregular shapes. ZIP codes can cross city or county boundaries, leading to potential inconsistencies when using them for spatial analysis.

ZCTAs, however, have well-defined and delineated boundaries. They are constructed by assigning census blocks to the most frequently occurring ZIP code within each block, resulting in contiguous and non-overlapping areas. ZCTAs are designed to align more closely with actual areas of population and geographic coverage.

#### Data:
The most reliable source for ZIP codes shapefiles are from ESRI files which are proprietary. 

ZCTAs shapefiles sources are publicly opened data taken from census.gov website  (https://www.census.gov/programs-surveys/geography/guidance/geo-areas/zctas.html). The file we use are TIGER/Line Shapefiles (https://www.census.gov/geographies/mapping-files/time-series/geo/tiger-line-file.html)

### TL;DR
ZIP codes and ZCTAs serve different purposes and have distinct characteristics. ZIP codes are primarily used for mail delivery, while ZCTAs are designed for statistical analysis. ZIP codes represent smaller, sometimes irregular areas, while ZCTAs are larger, well-defined areas. When it comes to spatial aggregation and statistical analysis, ZCTAs are generally preferred due to their consistency and suitability for demographic and geographic comparisons.

# Zip codes coverage investigation - Year 2018

We look at beneficiaries enrolled in 2018. Enrollments are previously harmonized as described in https://github.com/NSAPH-Data-Processing/mbsf_mortality_denom#readme

In [ ]:
## Load packages ----
import numpy as np
import pandas as pd
import duckdb
import itertools
import geopandas as gpd

In [ ]:
enroll_prefix = "../data/input/dw_medicare_bouzinier/enroll"

In [ ]:
# Read enrollment beneficiaries ----
conn = duckdb.connect()
query = f"""
    SELECT 
        zip, 
        COUNT(bene_id) as n_bene 
    FROM
        '{enroll_prefix}_2018.parquet' 
    GROUP BY zip
    ORDER BY zip
"""

bene_df = conn.execute(query).fetchdf()

print(bene_df.dtypes)
print(bene_df.head())
print(bene_df.shape)
print(bene_df.n_bene.sum())

In [ ]:
## Read enrollment zipcodes ----
df_zipcode_enrollments_dorieh = bene_df
df_zipcode_enrollments_dorieh["zip"] = df_zipcode_enrollments_dorieh["zip"].astype(int)

df_zipcode_enrollments_dorieh.head()

In [ ]:
## Read pobox csv file ----
csv_directory = "../data/input/Zipcode_Info/pobox_csv/ESRI18USZIP5_POINT_WGS84_POBOX.csv"
pobox_df = pd.read_csv(csv_directory)
pobox_df = pobox_df.rename(columns={"ZIP": "zip"})
pobox_df["zip"] = pobox_df["zip"].astype(int)

pobox_df.head()

In [ ]:
# Read ESRI shapefile into a geopandas dataframe
shapefile_path = "../data/input/Zipcode_Info/polygon/ESRI18USZIP5_POLY_WGS84.shp"
gdf = gpd.read_file(shapefile_path)
gdf.head()

In [ ]:
# set of distinct ZIP in the ESRI shapefile
set_of_distinct_ZIP_esri_shapefile = set()
set_of_distinct_ZIP_esri_shapefile.update(gdf["ZIP"].unique())
list_of_distinct_ZIP = list(set_of_distinct_ZIP_esri_shapefile)
esri_shapefile_df = pd.DataFrame({'zip': list_of_distinct_ZIP})
esri_shapefile_df["zip"] = esri_shapefile_df["zip"].astype(int)

esri_shapefile_df.head()

In [ ]:
print("Total number of unique zipcode in Patients Enrollment data: ", len(df_zipcode_enrollments_dorieh["zip"].unique()))
print("Total number of unique zipcode in PO Box data: ",len(pobox_df["zip"].unique()))
print("Total number of unique zipcode in ESRI Shapefile data: ", len(gdf["ZIP"].unique()))

### Data Cleaning

In [ ]:
# Determine whether there are inf or nan values in the dataframe
print(np.isinf(df_zipcode_enrollments_dorieh.zip).sum())
print(df_zipcode_enrollments_dorieh.isnull().sum())

# Replace infinite values with NaN and remove rows with NaN values
#df_zipcode_enrollments_dorieh.replace([np.inf, -np.inf], np.nan, inplace=True)
#df_zipcode_enrollments_dorieh.dropna(inplace=True)

In [ ]:
# Determine whether there are inf or nan values in the dataframe
print(np.isinf(pobox_df.zip).sum())
print(pobox_df.isnull().sum())

# Replace infinite values with NaN and remove rows with NaN values
#pobox_df.replace([np.inf, -np.inf], np.nan, inplace=True)
#pobox_df.dropna(inplace=True)


In [ ]:
# Determine whether there are inf or nan values in the dataframe
print(np.isinf(esri_shapefile_df.zip).sum())
print(esri_shapefile_df.isnull().sum())

# Replace infinite values with NaN and remove rows with NaN values
#esri_shapefile_df.replace([np.inf, -np.inf], np.nan, inplace=True)
#esri_shapefile_df.dropna(inplace=True)

In [ ]:
# Add "zipcode_in_enrollments" column to df_zipcode_enrollments_dorieh
df_zipcode_enrollments_dorieh = df_zipcode_enrollments_dorieh.assign(zipcode_in_enrollments=1)

# Add "zipcode_in_pobox" column to df_zipcode_enrollments_dorieh
df_zipcode_pobox = (
    pobox_df[['zip']]
    .assign(zipcode_in_pobox=1)
)

# Add "zipcode_in_esri" column to gdf
esri_shapefile_df = esri_shapefile_df.assign(zipcode_in_esri=1)

# Perform outer join on the "zip" column
zip_df = pd.merge(df_zipcode_enrollments_dorieh, df_zipcode_pobox, on="zip", how="outer")
zip_df = pd.merge(zip_df, esri_shapefile_df, on="zip", how="outer")

zip_df["zipcode_in_enrollments"] = zip_df["zipcode_in_enrollments"].fillna(0)
zip_df["zipcode_in_pobox"] = zip_df["zipcode_in_pobox"].fillna(0)
zip_df["zipcode_in_esri"] = zip_df["zipcode_in_esri"].fillna(0)

zip_df.head()


In [ ]:
print(f"total number of zipcodes: {zip_df.shape[0]}")
print(f"total number of beneficiaries: {zip_df.n_bene.sum()}")

### Add UDS Mapper crosswalk

Source: https://udsmapper.org/zip-code-to-zcta-crosswalk/

In [ ]:
# Specify the file path
file_path = "../data/input/aux/ZIPCodetoZCTACrosswalk2018UDS.xlsx"

# Read the Excel file into a DataFrame
df_UDS = pd.read_excel(file_path)
df_UDS.head()

In [ ]:
df_UDS.zip_join_type.unique()

In [ ]:
print(f"missing zipcode {df_UDS.ZIP_CODE.isnull().sum()}")
print(f"missing ZCTA {df_UDS.ZCTA.isnull().sum()}")

In [ ]:
selected_rows = df_UDS[(df_UDS['zip_join_type'] == 'populated ZCTA, missing zip') |
                       (df_UDS['zip_join_type'] == 'territory zip, no ZCTA available')]
selected_rows

In [ ]:
# Convert to integer
df_UDS['ZIP_CODE'] = df_UDS['ZIP_CODE'].astype(int)
df_UDS['ZCTA'] = pd.to_numeric(df_UDS['ZCTA'], errors='coerce', downcast='integer')

# Summary of number of zipcodes by ZCTA
df_UDS.groupby('ZCTA')['ZIP_CODE'].nunique().value_counts()

In [ ]:
# Summary of number of ZCTA by zipcodes
df_UDS.groupby('ZIP_CODE')['ZCTA'].nunique().value_counts()

In [ ]:
# An example of a zcta with multiple zipcodes
df_UDS[df_UDS.ZCTA == 79925]

In [ ]:
# Add "zipcode_in_UDS" column to zip_df
df_zipcode_UDS = (
    df_UDS[['ZIP_CODE']]
    .rename(columns={'ZIP_CODE': 'zip'})
    .assign(zipcode_in_UDS=1)
)

# Perform outer join on the "zip" column
zip_df = pd.merge(zip_df, df_zipcode_UDS, on="zip", how="outer")
zip_df["zipcode_in_UDS"] = zip_df["zipcode_in_UDS"].fillna(0)

zip_df.head()

### Add Census Reporter Crosswalk 

Source:https://github.com/censusreporter/acs-aggregate/blob/master/crosswalks/zip_to_zcta/ZIP_ZCTA_README.md 

Processing notebook: https://github.com/censusreporter/acs-aggregate/blob/master/crosswalks/zip_to_zcta/build_crosswalk.ipynb 

Mapping ZIP Codes to ZCTAs can be a challenging task due to their frequent changes and the lack of authoritative documentation in any known public resource. To tackle this, we will combine information from two sources: GeoNames and a dataset called ZIP Code Business Patterns (ZBP). By doing so, we aim to create an extensive list of potential ZIP Codes that can be accurately mapped to ZCTAs.

They used a dataset from GeoNames as their master list. For a comprehensive list of ZCTAs, they used the ZCTA file from the 2017 Gazetteer files. However, for this version, they used the Census's ZCTA ESRI Shapefile, which is just the Gazetteer plus geographic boundaries.



In [ ]:
# Specify the file path
file_path = "../data/input/aux/zip_zcta_xref.csv"

# Read the Excel file into a DataFrame
df_geoNames = pd.read_csv(file_path)
df_geoNames.head()

In [ ]:
# Convert to integer
df_geoNames['zip_code'] = df_geoNames['zip_code'].astype(int)
df_geoNames['zcta'] = pd.to_numeric(df_geoNames['zcta'], errors='coerce', downcast='integer')

# Summary of number of zipcodes by ZCTA
df_geoNames.groupby('zcta')['zip_code'].nunique().value_counts()

In [ ]:
# Summary of number of ZCTA by zipcodes
df_geoNames.groupby('zip_code')['zcta'].nunique().value_counts()

In [ ]:
# Add "zipcode_in_UDS" column to zip_df
df_zipcode_geonames = (
    df_geoNames[['zip_code']]
    .rename(columns={'zip_code': 'zip'})
    .assign(zipcode_in_geonames=1)
)

# Perform outer join on the "zip" column
zip_df = pd.merge(zip_df, df_zipcode_geonames, on="zip", how="outer")
zip_df["zipcode_in_geonames"] = zip_df["zipcode_in_geonames"].fillna(0)

zip_df.head()

## Enrollments summary

In [ ]:
def generate_table_summary(df, column_names):
    # Initialize summary data dictionary
    summary_data = {}

    for column in column_names:
        total_intersection = len(set(df[df[column] == 1.0]['zip']))
        sum_beneficiaries = df[df[column] == 1.0]['n_bene'].sum()

        # Create summary key based on column names
        key = f"total zip {column}"

        # Add key-value pair to summary data
        summary_data[key] = [total_intersection, sum_beneficiaries]

    for column1, column2 in itertools.combinations(column_names, 2):
        # Total intersection (column1, column2)
        total_intersection = len(set(df[df[column1] == 1.0][df[column2] == 1.0]['zip']))
        sum_beneficiaries = df[df[column1] == 1.0][df[column2] == 1.0]['n_bene'].sum()

        # Create summary key based on column names
        key = f"({column1}, {column2})"

        # Add key-value pair to summary data
        summary_data[key] = [total_intersection, sum_beneficiaries]
    
    # Create a summary DataFrame
    summary_df = pd.DataFrame.from_dict(summary_data, orient='index', columns=['Total Intersection', 'Total Beneficiaries'])

    return summary_df

In [ ]:
### Define the column names for the matrix-like summary
column_names = ['zipcode_in_enrollments', 'zipcode_in_pobox', 'zipcode_in_esri', 'zipcode_in_UDS', 'zipcode_in_geonames']

# Generate the table summary
summary_table = generate_table_summary(zip_df, column_names)

# Print the table summary
summary_table 

In [ ]:
def generate_table_summary(df, column_names):
    # Initialize summary data dictionary
    summary_data = {}

    for column in column_names:
        total_intersection = len(set(df[df[column] == 1.0]['zip']))
        sum_beneficiaries = df[df[column] == 1.0]['n_bene'].sum()

        # Create summary key based on column names
        key = f"total zip {column}"

        # Add key-value pair to summary data
        summary_data[key] = [total_intersection, sum_beneficiaries]

    for column1, column2 in itertools.product(column_names, repeat=2):
        # Total difference (column1, column2)
        total_difference = len(set(df[df[column1] == 1.0][df[column2] == 0.0]['zip']))
        sum_beneficiaries = df[df[column1] == 1.0][df[column2] == 0.0]['n_bene'].sum()

        # Create summary key based on column names
        key = f"({column1}, {column2})"

        # Add key-value pair to summary data
        summary_data[key] = [total_difference, sum_beneficiaries]
    
    # Create a summary DataFrame
    summary_df = pd.DataFrame.from_dict(summary_data, orient='index', columns=['Total Difference', 'Total Beneficiaries'])

    return summary_df

In [ ]:
### Define the column names for the matrix-like summary
column_names = ['zipcode_in_enrollments', 'zipcode_in_pobox', 'zipcode_in_esri', 'zipcode_in_UDS', 'zipcode_in_geonames']

# Generate the table summary
summary_table = generate_table_summary(zip_df, column_names)

# Print the table summary
summary_table 